In [15]:
import json
import re
from pathlib import Path

from IPython.display import display, Markdown
import matplotlib.pyplot as plt

TASKS = [
    "niah_single_1",
    "niah_single_2",
    "niah_single_3",
    "niah_multikey_1",
    "niah_multikey_2",
    "niah_multikey_3",
    "niah_multiquery",
    "niah_multivalue",
    "ruler_vt",
    "ruler_cwe",
    "ruler_fwe",
    "ruler_qa_squad",
    "ruler_qa_hotpot",
]

# 只有这些 task 允许读取/绘制 4096
TASKS_ALLOW_4096 = {"niah_single_1", "niah_single_2", "niah_single_3"}

ROOT = Path("results_pretrain")
BASELINE = "base_rope_42"

# acc[(exp_name, task, length)] = value
acc = {}

for exp_dir in sorted([p for p in ROOT.iterdir() if p.is_dir()]):
    exp = exp_dir.name
    for f in sorted(exp_dir.glob("*_eval.txt")):
        task = f.name[: -len("_eval.txt")]
        if task not in TASKS:
            continue

        try:
            data = json.loads(f.read_text(encoding="utf-8"))
        except Exception as e:
            print(f"[WARN] parse failed: {f}, {e}")
            continue

        metrics = data.get(task)
        if metrics is None:
            if any(re.fullmatch(r"\d+,none", str(k)) for k in data.keys()):
                metrics = data
            else:
                continue

        for k, v in metrics.items():
            m = re.fullmatch(r"(\d+),none", str(k))
            if not m or not isinstance(v, (int, float)):
                continue
            length = int(m.group(1))
            # 非 niah_single_* 直接忽略 4096
            if length == 4096 and task not in TASKS_ALLOW_4096:
                continue
            acc[(exp, task, length)] = float(v)

# 实验顺序：baseline 最左
all_exps = sorted({k[0] for k in acc.keys()})
if BASELINE in all_exps:
    exps = [BASELINE] + [e for e in all_exps if e != BASELINE]
else:
    print(f"[WARN] BASELINE={BASELINE} 不在结果里，按名字排序")
    exps = all_exps

def length_key(L):
    pref = {1024: 0, 2048: 1, 4096: 2, 8192: 3}
    return (pref.get(L, 99), L)

rows = []
for task in TASKS:
    lengths = sorted({L for (e, t, L) in acc.keys() if t == task}, key=length_key)
    for L in lengths:
        vals = [acc.get((e, task, L)) for e in exps]
        rows.append((task, L, vals))

if not rows:
    raise SystemExit("没有找到可用的 acc 数据")

# ---------- Markdown 表格 ----------
lines = []
lines.append("")
lines.append("| Task | Context | " + " | ".join(exps) + " |")
lines.append("|:---:|:---:|" + "|".join([":---:"] * len(exps)) + "|")

for task, L, vals in rows:
    cells = ["N/A" if v is None else f"{v:.3f}" for v in vals]
    lines.append(f"| {task} | {L//1000}K | " + " | ".join(cells) + " |")

print("\n".join(lines))
display(Markdown("\n".join(lines)))


| Task | Context | base_rope_42 | base_nope_w128d16r0h1g1_42 |
|:---:|:---:|:---:|:---:|
| niah_single_1 | 1K | 0.614 | 0.890 |
| niah_single_1 | 2K | 0.004 | 0.722 |
| niah_single_1 | 4K | 0.000 | 0.314 |
| niah_single_2 | 1K | 0.612 | 0.664 |
| niah_single_2 | 2K | 0.000 | 0.038 |
| niah_single_2 | 4K | 0.000 | 0.000 |
| niah_single_3 | 1K | 0.090 | 0.180 |
| niah_single_3 | 2K | 0.000 | 0.014 |
| niah_single_3 | 4K | 0.000 | 0.000 |
| niah_multikey_1 | 1K | 0.198 | 0.200 |
| niah_multikey_1 | 2K | 0.118 | 0.194 |
| niah_multiquery | 1K | 0.104 | 0.186 |
| niah_multiquery | 2K | 0.029 | 0.179 |
| ruler_qa_squad | 1K | 0.148 | 0.191 |
| ruler_qa_squad | 2K | 0.078 | 0.091 |



| Task | Context | base_rope_42 | base_nope_w128d16r0h1g1_42 |
|:---:|:---:|:---:|:---:|
| niah_single_1 | 1K | 0.614 | 0.890 |
| niah_single_1 | 2K | 0.004 | 0.722 |
| niah_single_1 | 4K | 0.000 | 0.314 |
| niah_single_2 | 1K | 0.612 | 0.664 |
| niah_single_2 | 2K | 0.000 | 0.038 |
| niah_single_2 | 4K | 0.000 | 0.000 |
| niah_single_3 | 1K | 0.090 | 0.180 |
| niah_single_3 | 2K | 0.000 | 0.014 |
| niah_single_3 | 4K | 0.000 | 0.000 |
| niah_multikey_1 | 1K | 0.198 | 0.200 |
| niah_multikey_1 | 2K | 0.118 | 0.194 |
| niah_multiquery | 1K | 0.104 | 0.186 |
| niah_multiquery | 2K | 0.029 | 0.179 |
| ruler_qa_squad | 1K | 0.148 | 0.191 |
| ruler_qa_squad | 2K | 0.078 | 0.091 |